# Data Science & Database Assignment: Session 12
## Topic: Common Table Expressions (CTE) & the WITH Clause

**Student Name:** Atharv Chaudhari  
**Course:** Data Science & Database Fundamentals  
**Database Name:** `cte_demo.db`  
**Core Concepts:** Common Table Expressions (`WITH`), Subquery vs CTE Readability, Multiple CTEs in a Single Query, Recursive CTEs (`WITH RECURSIVE`), Query Refactoring & Optimization  
**Status:** Completed & Pre-Executed  

---

### What is a Common Table Expression (CTE)?
A **Common Table Expression (CTE)** defines a temporary, named result set that exists only within the execution scope of a single SQL statement (`SELECT`, `INSERT`, `UPDATE`, or `DELETE`).

```text
+---------------------+-------------------------------------------------------------------------------+
| Aspect              | Subquery (Derived Table)            | Common Table Expression (CTE)           |
+---------------------+-------------------------------------+-----------------------------------------+
| Reading Order       | Inside-out (nested within FROM)     | Top-to-bottom (defined before SELECT)   |
| Reusability         | Cannot be referenced twice (must duplicate) | Can be referenced multiple times in joins |
| Recursion           | Does NOT support recursion          | Supports WITH RECURSIVE for trees/dates |
| Debugging           | Difficult to isolate inner queries  | Easy to test individual CTE blocks      |
+---------------------+-------------------------------------+-----------------------------------------+
```


## Task 1: Basic CTE with WITH Clause (Flipkart/Myntra Top-Rated Products)
> **Problem Statement:** Create a CTE using the `WITH` clause to select all products with a rating above 4.5 from a `Products` table, similar to how Flipkart or Myntra might highlight top-rated items.

### Conceptual Explanation
By using `WITH TopRatedProducts AS (...)`, we isolate the business logic that defines what constitutes a "Top Rated" item before deciding how to present or sort the final catalog.


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('cte_demo.db')
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS Products;")
cursor.execute("""
CREATE TABLE Products (
    id INT PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    category VARCHAR(50) NOT NULL,
    price DECIMAL(10, 2) NOT NULL,
    rating DECIMAL(2, 1) NOT NULL
);
""")

products_data = [
    (1, 'Sony WH-1000XM5 Wireless Headphones', 'Electronics', 28990.00, 4.8),
    (2, 'Apple iPhone 15 Pro', 'Electronics', 129900.00, 4.7),
    (3, 'Nike Air Jordan 1 Retro', 'Footwear', 11995.00, 4.9),
    (4, "Levi's 511 Slim Fit Jeans", 'Fashion', 2999.00, 4.2),
    (5, 'Puma Mens Running Shoes', 'Footwear', 3499.00, 4.1),
    (6, 'Samsung Galaxy S24 Ultra', 'Electronics', 129999.00, 4.6),
    (7, 'Fossil Gen 6 Smartwatch', 'Accessories', 14995.00, 4.0),
    (8, 'Ray-Ban Wayfarer Classic', 'Accessories', 8590.00, 4.6)
]
cursor.executemany("INSERT INTO Products VALUES (?, ?, ?, ?, ?);", products_data)
conn.commit()

task1_query = """
WITH TopRatedProducts AS (
    SELECT 
        id AS product_id,
        name AS product_name,
        category,
        price,
        rating
    FROM Products
    WHERE rating > 4.5
)
SELECT * FROM TopRatedProducts
ORDER BY rating DESC, price ASC;
"""

df_t1 = pd.read_sql_query(task1_query, conn)
display(df_t1)


,product_id,product_name,category,price,rating
0,3,Nike Air Jordan 1 Retro,Footwear,11995,4.9
1,1,Sony WH-1000XM5 Wireless Headphones,Electronics,28990,4.8
2,2,Apple iPhone 15 Pro,Electronics,129900,4.7
3,8,Ray-Ban Wayfarer Classic,Accessories,8590,4.6
4,6,Samsung Galaxy S24 Ultra,Electronics,129999,4.6


## Task 2: Subquery vs CTE Comparison (Ahmedabad Restaurants with Delivery < 50)
> **Problem Statement:** Rewrite a query that finds all restaurants in 'Ahmedabad' with delivery charges under 50 from a `Restaurants` table, first using a subquery and then using a CTE. Compare both queries for readability.  
> *Hint: Focus on making the CTE version cleaner and easier to understand.*

---

### Architectural & Readability Comparison
1. **Flow of Thought:** In the CTE, the developer reads:
   *"First, take Ahmedabad restaurants as a named dataset. Second, filter that dataset for delivery under 50."*
   In the subquery, the developer is forced to scan past the outer `SELECT` into the inner `FROM (...)`, read the inner query, and then jump back outward.
2. **Maintenance:** When requirements evolve (e.g., adding vegetarian filter or joining review scores), the CTE can be modularly expanded without deep nesting.


In [2]:
cursor.execute("DROP TABLE IF EXISTS Restaurants;")
cursor.execute("""
CREATE TABLE Restaurants (
    id INT PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    city VARCHAR(50) NOT NULL,
    cuisine VARCHAR(50) NOT NULL,
    delivery_charge DECIMAL(5, 2) NOT NULL,
    rating DECIMAL(2, 1) NOT NULL
);
""")

restaurants_data = [
    (1, 'Manek Chowk Delights', 'Ahmedabad', 'Street Food', 30.00, 4.6),
    (2, 'Agashiye - House of MG', 'Ahmedabad', 'Gujarati Thali', 40.00, 4.8),
    (3, 'Swati Snacks', 'Ahmedabad', 'Chaat & Snacks', 45.00, 4.5),
    (4, 'Gordhan Thal', 'Ahmedabad', 'Gujarati Thali', 60.00, 4.7),
    (5, 'Havmor Restaurant', 'Ahmedabad', 'North Indian', 55.00, 4.3),
    (6, 'Truffles', 'Bangalore', 'Continental', 35.00, 4.7),
    (7, 'Britannia & Co', 'Mumbai', 'Parsi', 40.00, 4.4)
]
cursor.executemany("INSERT INTO Restaurants VALUES (?, ?, ?, ?, ?, ?);", restaurants_data)
conn.commit()

print("--- Version A: Derived Table Subquery ---")
query_subquery = """
SELECT restaurant_name, city, cuisine, delivery_charge, rating
FROM (
    SELECT name AS restaurant_name, city, cuisine, delivery_charge, rating
    FROM Restaurants
    WHERE city = 'Ahmedabad'
) AS AhmedabadRestaurants
WHERE delivery_charge < 50.00
ORDER BY delivery_charge ASC;
"""
display(pd.read_sql_query(query_subquery, conn))

print("--- Version B: Clean Common Table Expression (CTE) ---")
query_cte = """
WITH AhmedabadRestaurants AS (
    SELECT name AS restaurant_name, city, cuisine, delivery_charge, rating
    FROM Restaurants
    WHERE city = 'Ahmedabad'
)
SELECT restaurant_name, city, cuisine, delivery_charge, rating
FROM AhmedabadRestaurants
WHERE delivery_charge < 50.00
ORDER BY delivery_charge ASC;
"""
display(pd.read_sql_query(query_cte, conn))


--- Version A: Derived Table Subquery ---


,restaurant_name,city,cuisine,delivery_charge,rating
0,Manek Chowk Delights,Ahmedabad,Street Food,30,4.6
1,Agashiye - House of MG,Ahmedabad,Gujarati Thali,40,4.8
2,Swati Snacks,Ahmedabad,Chaat & Snacks,45,4.5


--- Version B: Clean Common Table Expression (CTE) ---


,restaurant_name,city,cuisine,delivery_charge,rating
0,Manek Chowk Delights,Ahmedabad,Street Food,30,4.6
1,Agashiye - House of MG,Ahmedabad,Gujarati Thali,40,4.8
2,Swati Snacks,Ahmedabad,Chaat & Snacks,45,4.5


## Task 3: Multiple CTEs in a Single Query (Instagram Top Users & Top Posts)
> **Problem Statement:** Using two CTEs in a single query, find the top 3 most-followed users and the top 3 most-liked posts from a `Users` and `Posts` table (think Instagram-style data). Output both lists in the same result set.

### Conceptual Explanation
SQL allows defining multiple comma-separated CTEs under a single `WITH` keyword:
```sql
WITH CTE_One AS (...),
     CTE_Two AS (...)
SELECT ... FROM CTE_One
UNION ALL
SELECT ... FROM CTE_Two;
```
Combining both lists using `UNION ALL` creates an executive dashboard view pairing top users and top posts in a unified schema.


In [3]:
cursor.execute("DROP TABLE IF EXISTS Posts;")
cursor.execute("DROP TABLE IF EXISTS Users;")

cursor.execute("""
CREATE TABLE Users (
    id INT PRIMARY KEY,
    username VARCHAR(50) NOT NULL,
    followers_count INT NOT NULL,
    account_status VARCHAR(20) DEFAULT 'Active'
);
""")

cursor.execute("""
CREATE TABLE Posts (
    id INT PRIMARY KEY,
    user_id INT NOT NULL,
    caption VARCHAR(255) NOT NULL,
    likes_count INT NOT NULL
);
""")

users_data = [
    (1, 'virat.kohli', 270000000, 'Active'),
    (2, 'priyanka.chopra', 91000000, 'Active'),
    (3, 'alia.bhatt', 85000000, 'Active'),
    (4, 'tech_reviewer', 500000, 'Active'),
    (5, 'daily_foodie', 250000, 'Active')
]
posts_data = [
    (101, 1, 'Historic victory at MCG! Proud of the boys.', 5800000),
    (102, 1, 'Match day ready. Always grateful.', 4200000),
    (103, 2, 'Unforgettable night at the Met Gala 2026.', 3900000),
    (104, 3, 'Behind the scenes from our new film set.', 3100000),
    (105, 4, 'Unboxing the future: New foldable phone review.', 180000),
    (106, 5, 'Best street food crawl in Ahmedabad!', 95000)
]
cursor.executemany("INSERT INTO Users VALUES (?, ?, ?, ?);", users_data)
cursor.executemany("INSERT INTO Posts VALUES (?, ?, ?, ?);", posts_data)
conn.commit()

task3_query = """
WITH TopFollowedUsers AS (
    SELECT 
        'Most Followed User' AS metric_category,
        username AS entity_name,
        followers_count AS metric_value
    FROM Users
    ORDER BY followers_count DESC
    LIMIT 3
),
TopLikedPosts AS (
    SELECT 
        'Most Liked Post' AS metric_category,
        caption AS entity_name,
        likes_count AS metric_value
    FROM Posts
    ORDER BY likes_count DESC
    LIMIT 3
)
SELECT * FROM TopFollowedUsers
UNION ALL
SELECT * FROM TopLikedPosts;
"""

df_t3 = pd.read_sql_query(task3_query, conn)
display(df_t3)


,metric_category,entity_name,metric_value
0,Most Followed User,virat.kohli,270000000
1,Most Followed User,priyanka.chopra,91000000
2,Most Followed User,alia.bhatt,85000000
3,Most Liked Post,Historic victory at MCG! Proud of the boys.,5800000
4,Most Liked Post,Match day ready. Always grateful.,4200000
5,Most Liked Post,Unforgettable night at the Met Gala 2026.,3900000


## Task 4: Recursive CTE for Next 7 Days (BookMyShow Movie Booking Dates)
> **Problem Statement:** Write a recursive CTE that generates a list of dates for the next 7 days starting from today, similar to how BookMyShow shows available dates for movie bookings.  
> *Hint: Use a base case for today and recursion to add one day at a time.*

---

### Anatomy of a Recursive CTE
A recursive CTE contains three distinct components:
1. **Anchor Member (Base Case):** Non-recursive initial row query (`DATE('now') AS booking_date, 1 AS day_number`).
2. **Recursive Member:** References the CTE itself and iteratively produces subsequent rows (`day_number + 1, DATE(booking_date, '+1 day')`).
3. **Termination Condition:** Stops the recursion once the desired limit is reached (`WHERE day_number < 7`).


In [4]:
task4_query = """
WITH RECURSIVE BookingDates AS (
    -- Base Case: Day 1 (Today)
    SELECT 
        1 AS day_number,
        DATE('now') AS booking_date
    UNION ALL
    -- Recursive Step: Add +1 day up to 7 days
    SELECT 
        day_number + 1,
        DATE(booking_date, '+1 day')
    FROM BookingDates
    WHERE day_number < 7
)
SELECT 
    day_number,
    booking_date AS iso_date,
    strftime('%d-%m-%Y', booking_date) AS formatted_date,
    CASE strftime('%w', booking_date)
        WHEN '0' THEN 'Sunday'
        WHEN '1' THEN 'Monday'
        WHEN '2' THEN 'Tuesday'
        WHEN '3' THEN 'Wednesday'
        WHEN '4' THEN 'Thursday'
        WHEN '5' THEN 'Friday'
        WHEN '6' THEN 'Saturday'
    END AS day_of_week
FROM BookingDates;
"""

df_t4 = pd.read_sql_query(task4_query, conn)
display(df_t4)


,day_number,iso_date,formatted_date,day_of_week
0,1,2026-10-27,27-10-2026,Tuesday
1,2,2026-10-28,28-10-2026,Wednesday
2,3,2026-10-29,29-10-2026,Thursday
3,4,2026-10-30,30-10-2026,Friday
4,5,2026-10-31,31-10-2026,Saturday
5,6,2026-11-01,01-11-2026,Sunday
6,7,2026-11-02,02-11-2026,Monday


## Task 5: Refactoring a Messy Query Using CTEs for Clarity & Maintainability
> **Problem Statement:** Given a messy SQL query that finds all users with more than 1000 followers from a `Users` table, refactor it to use a CTE for better clarity and maintainability.

---

### Comparison of Before and After

#### 1. Messy Original Code (Inside-Out Nested Subqueries & Redundant Filters)
```sql
SELECT u.id, u.username, u.followers_count, u.account_status
FROM (
    SELECT * FROM (
        SELECT id, username, followers_count, account_status
        FROM Users
        WHERE account_status = 'Active'
    ) AS active_sub
    WHERE active_sub.followers_count > 1000
) AS u
WHERE u.id IN (
    SELECT user_id FROM (
        SELECT id AS user_id, followers_count FROM Users WHERE followers_count > 1000
    ) AS follower_sub
)
ORDER BY u.followers_count DESC;
```

#### 2. Clean CTE Refactoring
```sql
WITH ActiveInfluencerUsers AS (
    SELECT 
        id AS user_id,
        username,
        followers_count,
        account_status
    FROM Users
    WHERE account_status = 'Active'
      AND followers_count > 1000
)
SELECT 
    user_id,
    username,
    followers_count,
    account_status
FROM ActiveInfluencerUsers
ORDER BY followers_count DESC;
```


In [5]:
print("--- 1. Output from Messy Original Query ---")
display(pd.read_sql_query("""
SELECT u.id, u.username, u.followers_count, u.account_status
FROM (
    SELECT * FROM (
        SELECT id, username, followers_count, account_status
        FROM Users
        WHERE account_status = 'Active'
    ) AS active_sub
    WHERE active_sub.followers_count > 1000
) AS u
WHERE u.id IN (
    SELECT user_id FROM (
        SELECT id AS user_id, followers_count FROM Users WHERE followers_count > 1000
    ) AS follower_sub
)
ORDER BY u.followers_count DESC;
""", conn))

print("--- 2. Output from Clean Refactored CTE Query ---")
display(pd.read_sql_query("""
WITH ActiveInfluencerUsers AS (
    SELECT 
        id AS user_id,
        username,
        followers_count,
        account_status
    FROM Users
    WHERE account_status = 'Active'
      AND followers_count > 1000
)
SELECT 
    user_id,
    username,
    followers_count,
    account_status
FROM ActiveInfluencerUsers
ORDER BY followers_count DESC;
""", conn))

conn.close()


--- 1. Output from Messy Original Query ---


,id,username,followers_count,account_status
0,1,virat.kohli,270000000,Active
1,2,priyanka.chopra,91000000,Active
2,3,alia.bhatt,85000000,Active
3,4,tech_reviewer,500000,Active
4,5,daily_foodie,250000,Active


--- 2. Output from Clean Refactored CTE Query ---


,user_id,username,followers_count,account_status
0,1,virat.kohli,270000000,Active
1,2,priyanka.chopra,91000000,Active
2,3,alia.bhatt,85000000,Active
3,4,tech_reviewer,500000,Active
4,5,daily_foodie,250000,Active


## Summary & Production Best Practices

1. **Readability First:** Use CTEs whenever a query contains more than one level of subqueries to restore a natural top-to-bottom reading hierarchy.
2. **Naming Conventions:** Give CTEs descriptive names reflecting their semantic business state (e.g., `ActiveInfluencerUsers`, `AhmedabadRestaurants`) rather than generic names like `T1` or `SubQ`.
3. **Recursive CTEs:** Indispensable for calendar generation, date ranges, hierarchical org charts (Manager-Employee), and graph path traversal without external looping scripts.
4. **Optimization:** Most modern database engines (PostgreSQL 12+, MySQL 8+, SQLite 3.39+, SQL Server) automatically inline non-recursive CTEs into the query optimizer's execution plan, ensuring clean code executes with zero performance penalty.
